## 1: quais municípios e órgãos negociam melhor (têm a maior economia entre o valor estimado e o homologado)?

In [0]:
%sql
SELECT
  m.municipio_nome,
  COUNT(*)                                  AS contratacoes_com_variacao,
  ROUND(percentile(f.economia_pct, 0.5), 1) AS economia_mediana_pct,
  ROUND(SUM(f.economia_valor) / 1e6, 2)     AS economia_total_milhoes
FROM workspace.gold.fato_contratacao f
JOIN workspace.gold.dim_municipio m USING (municipio_codigo_ibge)
WHERE f.estimado_valido AND f.tem_homologado AND NOT f.valor_identico
GROUP BY m.municipio_nome
HAVING COUNT(*) >= 30
ORDER BY economia_mediana_pct DESC;

municipio_nome,contratacoes_com_variacao,economia_mediana_pct,economia_total_milhoes
Aurora do Pará,33,38.5,19.52
Marabá,251,31.9,189.33
Santarém,145,30.3,9817.89
Bragança,105,30.0,118.5
Ananindeua,43,27.5,11.26
Barcarena,55,25.0,97.65
Novo Progresso,34,23.5,22.93
Rondon do Pará,45,23.2,7.24
Altamira,81,23.1,57.19
Belém,1652,23.0,-28940.59


Entre os 34 municípios com pelo menos 30 contratações, a mediana de economia varia de 0,4% em Ipixuna do Pará a 38,5% em Aurora do Pará. Marabá e Santarém dão os resultados mais confiáveis, com 257 contratações a 31,9% e 145 a 30,3%, porque têm volume suficiente para que o número não dependa de poucos casos.

Belém tem 1.652 contratações e mediana de 23%, mas a soma da economia é negativa em quase R$ 29 bilhões. Na maior parte dos processos o município fecha abaixo do valor estimado, e alguns contratos de valor muito alto fecham acima o bastante para inverter o total.

As somas de valor estão comprometidas por registros com valores impossíveis, como Itaituba com R$ 360 bilhões de economia. A resposta usa a mediana, e o total aparece apenas como indicativo.

## 2: a taxa de homologação depende mais da modalidade ou do valor da contratação?

In [0]:
%sql
WITH base AS (
  SELECT
    f.tem_homologado,
    d.modalidade_nome,
    NTILE(4) OVER (ORDER BY f.valor_total_estimado) AS quartil_valor
  FROM workspace.gold.fato_contratacao f
  JOIN workspace.gold.dim_modalidade d USING (modalidade_id)
  WHERE f.estimado_valido
    AND f.data_publicacao < '2026-06-01'  
),
por_modalidade AS (
  SELECT
    'Modalidade' AS fator,
    modalidade_nome AS grupo,
    COUNT(*) AS contratacoes,
    ROUND(100.0 * SUM(CASE WHEN tem_homologado THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_homologacao_pct
  FROM base
  GROUP BY modalidade_nome
  HAVING COUNT(*) >= 30
),
por_valor AS (
  SELECT
    'Faixa de valor' AS fator,
    CONCAT('Quartil ', quartil_valor) AS grupo,
    COUNT(*) AS contratacoes,
    ROUND(100.0 * SUM(CASE WHEN tem_homologado THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_homologacao_pct
  FROM base
  GROUP BY quartil_valor
)
SELECT
  fator,
  grupo,
  contratacoes,
  taxa_homologacao_pct,
  ROUND(MAX(taxa_homologacao_pct) OVER (PARTITION BY fator)
      - MIN(taxa_homologacao_pct) OVER (PARTITION BY fator), 1) AS amplitude_do_fator
FROM (SELECT * FROM por_modalidade UNION ALL SELECT * FROM por_valor)
ORDER BY fator, taxa_homologacao_pct DESC;

fator,grupo,contratacoes,taxa_homologacao_pct,amplitude_do_fator
Faixa de valor,Quartil 1,2417,77.9,19.9
Faixa de valor,Quartil 4,2416,64.9,19.9
Faixa de valor,Quartil 3,2416,60.1,19.9
Faixa de valor,Quartil 2,2417,58.0,19.9
Modalidade,Dispensa,3605,71.0,51.6
Modalidade,Pregão - Eletrônico,3222,66.6,51.6
Modalidade,Concorrência - Eletrônica,852,64.1,51.6
Modalidade,Inexigibilidade,1723,57.3,51.6
Modalidade,Pregão - Presencial,73,32.9,51.6
Modalidade,Credenciamento,165,19.4,51.6


A modalidade influencia mais. A amplitude entre modalidades é de 51,6 pontos percentuais, contra 19,9 pontos entre as faixas de valor.

A diferença vem do tipo de processo. Dispensa e pregão eletrônico, com 71,0% e 66,6%, têm rito rápido e padronizado, e chegam ao fim em poucas semanas. O credenciamento, com 19,4%, funciona como chamamento aberto por longos períodos, então a ausência de homologação nele é esperada. O pregão presencial aparece com 32,9%, mas tem só 73 registros, base pequena demais para conclusão.

O valor também afeta o resultado, embora de forma irregular. O quartil mais baixo tem a maior taxa, 77,9%. O segundo cai para 58,0% e depois a taxa volta a subir até 64,9% no quartil mais alto. Não há relação direta entre valor maior e menor conclusão.

## 3: como as modalidades se distribuem em quantidade e em valor, e qual o peso da contratação direta?

In [0]:
%sql
SELECT
  d.forma_contratacao,
  d.modalidade_nome,
  COUNT(*)                                                     AS contratacoes,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)           AS pct_contratacoes,
  ROUND(SUM(f.valor_total_estimado) / 1e6, 2)                  AS valor_estimado_milhoes,
  ROUND(100.0 * SUM(f.valor_total_estimado) / SUM(SUM(f.valor_total_estimado)) OVER (), 1) AS pct_valor
FROM workspace.gold.fato_contratacao f
JOIN workspace.gold.dim_modalidade d USING (modalidade_id)
GROUP BY d.forma_contratacao, d.modalidade_nome
ORDER BY contratacoes DESC;

forma_contratacao,modalidade_nome,contratacoes,pct_contratacoes,valor_estimado_milhoes,pct_valor
Contratação direta,Dispensa,6438,36.9,3632.06,0.8
Licitação,Pregão - Eletrônico,5566,31.9,34826.71,8.1
Contratação direta,Inexigibilidade,3534,20.3,373201.76,86.4
Licitação,Concorrência - Eletrônica,1487,8.5,13606.95,3.2
Licitação,Credenciamento,262,1.5,5501.65,1.3
Licitação,Pregão - Presencial,100,0.6,698.46,0.2
Licitação,Concorrência - Presencial,29,0.2,311.78,0.1
Licitação,Leilão - Eletrônico,7,0.0,110.73,0.0
Licitação,Concurso,5,0.0,0.02,0.0
Licitação,Pré-qualificação,2,0.0,0.0,0.0


Dispensa e inexigibilidade somam 57,2% das contratações do período, com 6.438 e 3.534 registros de um total aproximado de 17.450. A maior parte das contratações públicas do estado não passa por licitação competitiva. O pregão eletrônico responde por 31,9%.

Em valor, a leitura exige cuidado. A inexigibilidade aparece com 86,4% do total financeiro, mas o número está dominado pelos mesmos registros anômalos da pergunta 1. Os R$ 373 bilhões atribuídos a ela superam em muito o orçamento anual do estado, o que indica erro de preenchimento em poucos registros.

O padrão da dispensa se sustenta: 36,9% das contratações e 0,8% do valor, uso compatível com o que a Lei 14.133 prevê para compras pequenas e frequentes. O pregão eletrônico tem perfil mais equilibrado, com 31,9% das contratações e 8,1% do valor.

## 4: como o volume e o valor das contratações se distribuem ao longo dos meses?

In [0]:
%sql
SELECT
  t.ano_mes,
  COUNT(*)                                    AS contratacoes,
  ROUND(SUM(f.valor_total_estimado) / 1e6, 2) AS valor_estimado_milhoes,
  ROUND(100.0 * SUM(CASE WHEN f.tem_homologado THEN 1 ELSE 0 END) / COUNT(*), 1) AS taxa_homologacao_pct
FROM workspace.gold.fato_contratacao f
JOIN workspace.gold.dim_tempo t ON f.data_publicacao = t.data
GROUP BY t.ano_mes
ORDER BY t.ano_mes;

ano_mes,contratacoes,valor_estimado_milhoes,taxa_homologacao_pct
2026-01,1630,6284.23,57.9
2026-02,1820,5534.46,62.2
2026-03,2391,18082.76,62.3
2026-04,2136,370844.45,65.2
2026-05,2266,9504.72,62.4
2026-06,2188,7701.82,59.6
2026-07,2457,5612.11,54.5
2026-08,2544,8325.6,42.3


O volume cresce ao longo do período, de 1.630 contratações em janeiro para 2.544 em agosto, alta de 56%. Parte disso é sazonalidade orçamentária e parte pode ser aumento da adesão dos órgãos ao PNCP.

A taxa de homologação sobe até 65,2% em abril e cai de forma contínua depois, chegando a 42,3% em agosto. A queda não indica piora na gestão. É o efeito do tempo de maturação dos processos, que justificou o corte temporal usado na pergunta 2: contratações recentes ainda não tiveram prazo para ser concluídas e registradas.

Abril também concentra o valor estimado anômalo de R$ 370 bilhões, provavelmente o mesmo registro que distorce a inexigibilidade na pergunta 3.

## Resumo Geral

O conjunto das respostas aponta três coisas:
-  A contratação direta responde por mais da metade dos processos no Pará, ainda que concentrada em valores baixos. 
- O que determina se uma contratação chega ao fim registrado é principalmente o rito escolhido, e não o valor envolvido. 
- O PNCP serve bem para contagens e medidas de tendência central, mas não para somas financeiras sem tratamento de outliers, além de ter uma parcela relevante de registros que só repetem o valor estimado no campo do homologado.